In [5]:
from dotenv import load_dotenv
load_dotenv("../.env")

import os
from pathlib import Path

# Gemini embedding model — same one we used during indexing
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Gemini LLM — this is what generates the final answer
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain's Chroma wrapper — to load our saved vector store
from langchain_chroma import Chroma

# PromptTemplate lets us write a reusable prompt with placeholders
from langchain_core.prompts import PromptTemplate

# RunnablePassthrough passes the user's question through the chain unchanged
from langchain_core.runnables import RunnablePassthrough

# StrOutputParser converts Gemini's response object into a plain string
from langchain_core.output_parsers import StrOutputParser

api_key = os.getenv("GOOGLE_API_KEY")
CHROMA_DIR = Path("../chroma_db")

print(f"API key loaded: {api_key[:8]}...")
print(f"ChromaDB path exists: {CHROMA_DIR.exists()}")


API key loaded: AQ.Ab8RN...
ChromaDB path exists: True


In [6]:
# Set up the same embedding model we used during indexing
# IMPORTANT: must be exactly the same model — otherwise the vectors won't match
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=api_key
)

# Load the existing ChromaDB vector store from disk
# This does NOT re-embed anything — it just opens the saved database
vectorstore = Chroma(
    collection_name="ml_papers",       # same collection name we used in Step 3
    persist_directory=str(CHROMA_DIR), # same folder we saved to in Step 3
    embedding_function=embeddings      # needed so Chroma knows how to embed new queries
)

# Create a retriever from the vector store
# k=5 means: return the 5 most relevant chunks for any given question
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

# Quick test — let's see what chunks come back for a sample question
test_docs = retriever.invoke("What is a residual connection?")
print(f"Retrieved {len(test_docs)} chunks\n")

# Print the source and page of each retrieved chunk
for doc in test_docs:
    print(f"Source: {doc.metadata['source']} | Page: {doc.metadata['page']}")



Retrieved 5 chunks

Source: resnet | Page: 3
Source: resnet | Page: 2
Source: resnet | Page: 2
Source: resnet | Page: 3
Source: resnet | Page: 2


In [7]:
# This is the heart of our prompt engineering
# {context} will be replaced with the retrieved chunks
# {question} will be replaced with the user's actual question

prompt_template = """
You are a helpful research assistant that explains machine learning papers 
to third-year undergraduate students in plain English.

Use ONLY the information provided in the context below to answer the question.
Do NOT use any outside knowledge.

For every piece of information you use, cite the source paper and page number 
in this format: [Paper: source_name, Page: X]

If the context contains mathematical notation or equations that are hard to read,
describe what the equation means in plain English instead of copying it directly.

If the context does not contain enough information to answer the question,
say: "I don't have enough information in the indexed papers to answer this question."

Context:
{context}

Question:
{question}

Answer (in plain English, with citations):
"""

# Create a LangChain PromptTemplate object from the string above
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]  # the two placeholders we defined
)

print("Prompt template ready.")
print(f"Input variables: {prompt.input_variables}")


Prompt template ready.
Input variables: ['context', 'question']


In [8]:
# Set up Gemini 2.5 Flash as the answering LLM
# temperature=0.2 means answers are mostly factual with a little flexibility
# (0 = fully deterministic, 1 = very creative — for research Q&A we want low temperature)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.2
)

# This helper function takes the list of retrieved Document objects
# and formats them into a single string that fits into the {context} placeholder
def format_docs(docs):
    formatted=[]
    for doc in docs:
        # Each chunk shows its source paper, page number, and text content
        chunk_text = f"[Paper: {doc.metadata['source']}, Page: {doc.metadata['page']}]\n{doc.page_content}"
        formatted.append(chunk_text)
    # Join all chunks with a separator so Gemini can clearly see where one ends and another begins
    return "\n\n---\n\n".join(formatted)

# Build the RAG chain using LangChain's pipe operator (|)
# Read this left to right — it shows exactly how data flows:
# 1. Take the question → retrieve relevant docs AND pass question through unchanged
# 2. Format the docs into a string, keep the question
# 3. Slot both into the prompt template
# 4. Send the filled prompt to Gemini
# 5. Parse Gemini's response into a plain string
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")


RAG chain built successfully.


In [9]:
# test
question = "What problem does the Transformer architecture solve that previous models couldn't?"

print(f"Question: {question}")
print("="*60)

# Invoke the full RAG chain — this triggers the entire pipeline
answer = rag_chain.invoke(question)

print(answer)

Question: What problem does the Transformer architecture solve that previous models couldn't?
The Transformer architecture solves two main problems that previous models faced:

1.  **Lack of Parallelization and Sequential Computation:** Previous models, particularly those based on recurrent networks, had a "sequential nature" that prevented parallel processing within training examples. This became especially problematic with longer sequences, as memory limitations restricted the ability to process multiple examples at once [Paper: attention_is_all_you_need, Page: 2]. The Transformer addresses this by "eschewing recurrence" and instead relying entirely on an attention mechanism, which "allows for significantly more parallelization" [Paper: attention_is_all_you_need, Page: 2]. This makes the Transformer "more parallelizable and requiring significantly less time to train" [Paper: attention_is_all_you_need, Page: 1].

2.  **Difficulty Learning Long-Range Dependencies:** Models like ConvS2S

In [16]:
import gradio as gr

# This is the function Gradio will call every time the user submits a question
# chat_history is a list of [user_message, bot_response] pairs — Gradio manages it automatically
def answer_question(question, chat_history):

    #Guard against empty questions
    if not question.strip():
        return "", chat_history
    
    # Run the question through our RAG chain from Cell 4
    # This triggers: embed question → retrieve chunks → fill prompt → Gemini → parse output
    answer = rag_chain.invoke(question)

    #appen the new question-answer pair to the chat history
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

    #return empty string to clear the input box, and the updated chat history
    return "", chat_history

#Gradio interface using Blocks
with gr.Blocks(title="ML Research Paper Chatbot") as demo:

    #title and description
    gr.Markdown( "# 📚 ML Research Paper Chatbot")
    gr.Markdown(
        "Ask questions about 12 landmark ML papers including Attention Is All You Need, "
        "BERT, ResNet, GANs, RAG, and more. Every answer is cited with the source paper and page."
    )

    with gr.Accordion("📄 Indexed Papers — click to see the full list", open=False):
        gr.Markdown("""
        | # | Paper | Year |
        |---|-------|------|
        | 1 | Attention Is All You Need (Transformer) | 2017 |
        | 2 | BERT | 2018 |
        | 3 | ResNet | 2015 |
        | 4 | GAN — Generative Adversarial Networks | 2014 |
        | 5 | Vision Transformer (ViT) | 2020 |
        | 6 | RAG — Retrieval-Augmented Generation | 2020 |
        | 7 | Word2Vec | 2013 |
        | 8 | Adam Optimizer | 2014 |
        | 9 | Batch Normalization | 2015 |
        | 10 | Dropout | 2014 |
        | 11 | Deep RL — Atari | 2015 |
        | 12 | Scaling Laws | 2020 |
        """)

    #the chat window - displays the conversation history
    chatbot = gr.Chatbot(
        label="Chat",
        height=500,         #height of the chat window in pixels
    )


    #A row layout to put the text box and button side by side
    with gr.Row():

        #The text input
        question_box = gr.Textbox(
            placeholder="e.g. What is self-attention and why does it matter?",
            label="Your Question",
            scale=8
        )

        submit_btn = gr.Button("Ask", variant="primary", scale=2)

    gr.Examples(
        examples=[
            "What problem does the Transformer architecture solve?",
            "How does BERT differ from GPT?",
            "What is a residual connection and why does it help?",
            "How do GANs work?",
            "What is the RAG architecture?",
            "Why does the Adam optimizer work better than SGD?",
        ],
        inputs=question_box   # clicking an example fills the question box
    )

    submit_btn.click(
        fn=answer_question,                     # function to call
        inputs= [question_box, chatbot],         # pass the question and chat history
        outputs=[question_box, chatbot]         # update the input box and chat window
    )

    #pressing Enter to submit
    question_box.submit(
        fn=answer_question,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    )

print("Gradio interface built. Ready to launch")





Gradio interface built. Ready to launch


In [17]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
